# 07 — Final Evaluation: Baseline BGE-M3 vs. Fine-Tuned LoRA Model

This notebook performs the **final held-out evaluation** of the embedding model.

Notebook 06 used the validation set to select the best LoRA checkpoint. In this
notebook, that selected checkpoint is evaluated on the **test set**, which has not
been used for training or checkpoint selection.

The central experimental question is:

> **Does domain-specific LoRA fine-tuning improve BGE-M3 retrieval on unseen
> Portuguese legal articles?**

## Experimental logic

```text
TRAIN
  ↓
learn LoRA parameters

VALIDATION
  ↓
select the best epoch

TEST
  ↓
final comparison
  ↓
Baseline BGE-M3 vs. BGE-M3 + LoRA
```

The test protocol must be identical for both models:

- same passage corpus;
- same test queries;
- same qrels;
- same maximum sequence length;
- same embedding normalisation;
- same similarity calculation;
- same Top-K;
- same retrieval metrics.

This isolates the model adaptation as the principal experimental difference.

## Outputs produced

The notebook saves:

- per-query baseline and LoRA results;
- global test metrics;
- semantic and reference results;
- results by legal document;
- absolute and relative metric changes;
- improved, unchanged and degraded semantic queries;
- qualitative ranking examples;
- a final experiment report.

## Important methodological rule

The test results should be inspected **only after the model and checkpoint have
already been selected**.

We do not use the test set to decide whether another epoch should be trained.

## 1. Imports

The final evaluation requires no optimiser and no training loss.

We need:

- Sentence Transformers to load and encode with BGE-M3;
- PEFT to attach the saved LoRA adapter;
- PyTorch for CUDA execution;
- NumPy for similarity calculations;
- pandas for result analysis;
- JSON for dataset and experiment metadata.

The notebook also records timings and explicitly releases model memory between
baseline and LoRA evaluation. This is useful on a 6 GB GPU.

In [1]:
import gc
import json
import math
import time
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from sentence_transformers import SentenceTransformer
from peft import PeftModel

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

C:\Users\user\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.13.0+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


## 2. Project paths

The test dataset comes from the same `dataset_v2` used throughout the experiment.

The selected LoRA adapter is loaded from:

```text
models/bge-m3-legal-pt-lora/best
```

The notebook also reads the fine-tuning metadata saved by notebook 06. This avoids
silently changing important settings such as the base model or sequence length.

Final results are written to:

```text
results/final_evaluation/
```

In [2]:
PROJECT_ROOT = Path("..")

DATASET_DIR = PROJECT_ROOT / "data" / "dataset_v2"
MODEL_DIR = PROJECT_ROOT / "models" / "bge-m3-legal-pt-lora"
BEST_ADAPTER_DIR = MODEL_DIR / "best"

FINETUNING_RESULTS_DIR = PROJECT_ROOT / "results" / "finetuning_lora"
FINETUNING_METADATA_PATH = FINETUNING_RESULTS_DIR / "finetuning_metadata.json"

RESULTS_DIR = PROJECT_ROOT / "results" / "final_evaluation"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PASSAGES_PATH = DATASET_DIR / "passages.jsonl"
TEST_QUERIES_PATH = DATASET_DIR / "test_queries.jsonl"
TEST_QRELS_PATH = DATASET_DIR / "test_qrels.jsonl"

print("Dataset      :", DATASET_DIR.resolve())
print("Best adapter :", BEST_ADAPTER_DIR.resolve())
print("Results      :", RESULTS_DIR.resolve())

Dataset      : C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\data\dataset_v2
Best adapter : C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\models\bge-m3-legal-pt-lora\best
Results      : C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\results\final_evaluation


## 3. Load the fine-tuning configuration

The final evaluation should reproduce the inference configuration used during
validation.

Instead of manually rewriting those values, we read them from
`finetuning_metadata.json`.

The most important fields are:

- base model;
- maximum sequence length;
- encoding batch size;
- best epoch;
- best validation metric.

The best epoch is reported for documentation only. The test set does **not**
participate in checkpoint selection.

In [3]:
with FINETUNING_METADATA_PATH.open("r", encoding="utf-8") as f:
    finetuning_metadata = json.load(f)

MODEL_NAME = finetuning_metadata["base_model"]
MAX_SEQ_LENGTH = finetuning_metadata["max_seq_length"]
ENCODE_BATCH_SIZE = finetuning_metadata["encode_batch_size"]

TOP_K = 10
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Base model          :", MODEL_NAME)
print("Maximum sequence    :", MAX_SEQ_LENGTH)
print("Encoding batch size :", ENCODE_BATCH_SIZE)
print("Selected epoch      :", finetuning_metadata["best_epoch"])
print(
    "Best validation MRR :",
    f'{finetuning_metadata["best_validation_metric"]:.4f}'
)
print("Evaluation device   :", DEVICE)

if not BEST_ADAPTER_DIR.exists():
    raise FileNotFoundError(
        f"Best LoRA adapter not found: {BEST_ADAPTER_DIR}"
    )

Base model          : BAAI/bge-m3
Maximum sequence    : 256
Encoding batch size : 4
Selected epoch      : 2
Best validation MRR : 0.9791
Evaluation device   : cuda


## 4. JSONL loader

The same deterministic JSONL loading strategy is used as in the previous
notebooks.

No malformed or missing test records should be silently ignored. The final metrics
are only meaningful if the complete held-out dataset is evaluated.

In [4]:
def read_jsonl(path):
    records = []

    with path.open("r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"Invalid JSON in {path.name}, line {line_number}: {exc}"
                )

    return records

## 5. Load the held-out test dataset

We load:

```text
passages.jsonl     → complete searchable corpus
test_queries.jsonl → held-out questions
test_qrels.jsonl   → ground-truth relevance judgements
```

The complete passage corpus is used because a realistic retrieval system must rank
the correct legal provision against all available passages, not merely against
passages from the test split.

The test queries themselves come only from articles assigned to the held-out test
partition.

In [5]:
passages = read_jsonl(PASSAGES_PATH)
test_queries = read_jsonl(TEST_QUERIES_PATH)
test_qrels = read_jsonl(TEST_QRELS_PATH)

passage_by_id = {
    passage["passage_id"]: passage
    for passage in passages
}

print("Passages     :", len(passages))
print("Test queries :", len(test_queries))
print("Test qrels   :", len(test_qrels))
print(
    "Query types  :",
    Counter(query["query_type"] for query in test_queries)
)

Passages     : 2088
Test queries : 561
Test qrels   : 953
Query types  : Counter({'semantic': 420, 'reference': 141})


## 6. Validate the test data

Before evaluating either model, we verify the test ground truth.

The checks ensure that:

- passage IDs are unique;
- test query IDs are unique;
- every qrel refers to a valid test query;
- every qrel refers to an existing passage;
- every test query has at least one relevant passage.

This prevents dataset errors from being mistaken for model errors.

In [6]:
passage_ids = [p["passage_id"] for p in passages]
query_ids = [q["query_id"] for q in test_queries]

assert len(passage_ids) == len(set(passage_ids)), "Duplicate passage IDs."
assert len(query_ids) == len(set(query_ids)), "Duplicate test query IDs."

query_id_set = set(query_ids)
passage_id_set = set(passage_ids)

relevant_test = defaultdict(set)

for qrel in test_qrels:
    assert qrel["query_id"] in query_id_set, (
        f'Unknown test query in qrels: {qrel["query_id"]}'
    )
    assert qrel["passage_id"] in passage_id_set, (
        f'Unknown passage in qrels: {qrel["passage_id"]}'
    )

    if qrel.get("relevance", 0) > 0:
        relevant_test[qrel["query_id"]].add(qrel["passage_id"])

queries_without_relevance = [
    qid for qid in query_ids
    if not relevant_test[qid]
]

assert not queries_without_relevance, (
    f"Queries without relevant passages: {queries_without_relevance[:5]}"
)

print("Test-data validation: OK")

Test-data validation: OK


## 7. Retrieval metrics

We use exactly the same metrics as in baseline evaluation and validation.

### Recall@K

Recall measures the proportion of relevant passages recovered within the first K
results.

### MRR@10

Mean Reciprocal Rank focuses on the position of the **first relevant passage**.

```text
rank 1 → 1.00
rank 2 → 0.50
rank 4 → 0.25
```

### nDCG@10

nDCG evaluates ranking quality while giving greater importance to relevant passages
near the top.

Using unchanged metric definitions is essential. Otherwise, differences between
notebooks could be caused by evaluation code rather than by the model.

In [7]:
def recall_at_k(ranked_ids, relevant_ids, k):
    if not relevant_ids:
        return 0.0

    return (
        len(set(ranked_ids[:k]) & relevant_ids)
        / len(relevant_ids)
    )


def reciprocal_rank_at_k(ranked_ids, relevant_ids, k):
    for rank, passage_id in enumerate(ranked_ids[:k], start=1):
        if passage_id in relevant_ids:
            return 1.0 / rank

    return 0.0


def ndcg_at_k(ranked_ids, relevant_ids, k):
    dcg = 0.0

    for rank, passage_id in enumerate(ranked_ids[:k], start=1):
        if passage_id in relevant_ids:
            dcg += 1.0 / math.log2(rank + 1)

    ideal_hits = min(len(relevant_ids), k)

    if ideal_hits == 0:
        return 0.0

    idcg = sum(
        1.0 / math.log2(rank + 1)
        for rank in range(1, ideal_hits + 1)
    )

    return dcg / idcg

## 8. General retrieval evaluation function

Unlike notebook 06, this function retains **per-query rankings** as well as
aggregate metrics.

For each model:

```text
passages → embeddings ─┐
                       ├→ cosine similarity → Top-K ranking
queries  → embeddings ─┘
```

The returned table stores:

- query ID and text;
- query type;
- source document;
- relevant passage IDs;
- ranked passage IDs;
- similarity scores;
- Recall@1/5/10;
- MRR@10;
- nDCG@10.

Keeping the detailed ranking allows us to compare not only whether the average
score improved, but **which individual queries improved or degraded**.

In [8]:
def evaluate_model(
    model,
    passages,
    queries,
    relevance_map,
    batch_size=ENCODE_BATCH_SIZE,
    top_k=TOP_K,
):
    start = time.time()

    passage_embeddings = model.encode(
        [passage["text"] for passage in passages],
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    query_embeddings = model.encode(
        [query["query"] for query in queries],
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    passage_embeddings = np.asarray(passage_embeddings, dtype=np.float32)
    query_embeddings = np.asarray(query_embeddings, dtype=np.float32)

    scores = query_embeddings @ passage_embeddings.T

    rows = []

    for query_index, query in enumerate(queries):
        row_scores = scores[query_index]
        candidate_count = min(top_k, len(passages))

        top_indices = np.argpartition(
            -row_scores,
            candidate_count - 1
        )[:candidate_count]

        top_indices = top_indices[
            np.argsort(-row_scores[top_indices])
        ]

        ranked_ids = [
            passages[index]["passage_id"]
            for index in top_indices
        ]

        ranked_scores = [
            float(row_scores[index])
            for index in top_indices
        ]

        relevant_ids = relevance_map[query["query_id"]]

        rows.append({
            "query_id": query["query_id"],
            "query": query["query"],
            "query_type": query.get("query_type"),
            "document": query.get("document"),
            "article_id": query.get("article_id"),
            "relevant_ids": sorted(relevant_ids),
            "ranked_ids": ranked_ids,
            "ranked_scores": ranked_scores,
            "recall_at_1": recall_at_k(ranked_ids, relevant_ids, 1),
            "recall_at_5": recall_at_k(ranked_ids, relevant_ids, 5),
            "recall_at_10": recall_at_k(ranked_ids, relevant_ids, 10),
            "mrr_at_10": reciprocal_rank_at_k(ranked_ids, relevant_ids, 10),
            "ndcg_at_10": ndcg_at_k(ranked_ids, relevant_ids, 10),
        })

    elapsed = time.time() - start

    del passage_embeddings, query_embeddings, scores
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return pd.DataFrame(rows), elapsed

## 9. Aggregation helpers

Per-query metrics are aggregated in three ways:

```text
ALL
SEMANTIC
REFERENCE
```

We also calculate results by legal document.

This is important because a single global average can hide two different effects:

- the model may improve semantic retrieval substantially;
- reference queries may already be easy for the baseline and change very little.

Since notebook 06 trained only on semantic queries, the semantic test subset is the
most direct measure of the intended adaptation.

In [9]:
METRIC_COLUMNS = [
    "recall_at_1",
    "recall_at_5",
    "recall_at_10",
    "mrr_at_10",
    "ndcg_at_10",
]


def aggregate_results(df, model_label):
    rows = []

    groups = [
        ("all", df),
        ("semantic", df[df["query_type"] == "semantic"]),
        ("reference", df[df["query_type"] == "reference"]),
    ]

    for query_group, subset in groups:
        if subset.empty:
            continue

        row = {
            "model": model_label,
            "query_group": query_group,
            "queries": len(subset),
        }

        for metric in METRIC_COLUMNS:
            row[metric] = float(subset[metric].mean())

        rows.append(row)

    return pd.DataFrame(rows)


def aggregate_by_document(df, model_label):
    rows = []

    for (document, query_type), subset in df.groupby(
        ["document", "query_type"],
        dropna=False
    ):
        row = {
            "model": model_label,
            "document": document,
            "query_type": query_type,
            "queries": len(subset),
        }

        for metric in METRIC_COLUMNS:
            row[metric] = float(subset[metric].mean())

        rows.append(row)

    return pd.DataFrame(rows)

## 10. Evaluate the original BGE-M3 baseline on the test set

We first evaluate the untouched pretrained model.

This creates the held-out baseline against which the selected LoRA model will be
compared.

No adapter is attached in this stage.

After evaluation, the baseline model is deleted from GPU memory before the adapted
model is loaded. This is important on a 6 GB GPU because keeping both models in
memory simultaneously would waste VRAM.

In [10]:
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

baseline_model = SentenceTransformer(
    MODEL_NAME,
    device=DEVICE
)

baseline_model.max_seq_length = MAX_SEQ_LENGTH

print("Running BASELINE test retrieval...")

baseline_results, baseline_seconds = evaluate_model(
    baseline_model,
    passages,
    test_queries,
    relevant_test,
)

print(f"Baseline evaluation time: {baseline_seconds:.1f} s")

baseline_summary = aggregate_results(
    baseline_results,
    "baseline"
)

display(baseline_summary)

del baseline_model
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 18352.43it/s]


Running BASELINE test retrieval...


Batches: 100%|██████████| 141/141 [00:03<00:00, 38.06it/s]


Baseline evaluation time: 64.9 s


,model,query_group,queries,recall_at_1,recall_at_5,recall_at_10,mrr_at_10,ndcg_at_10
0,baseline,all,561,0.526084,0.724228,0.789835,0.704021,0.701431
1,baseline,semantic,420,0.641660,0.837004,0.885681,0.824853,0.818472
2,baseline,reference,141,0.181816,0.388298,0.504334,0.344095,0.352798


## 11. Load BGE-M3 with the selected LoRA adapter

The adapted model is reconstructed from two components:

```text
BAAI/bge-m3
      +
models/bge-m3-legal-pt-lora/best
      =
fine-tuned embedding model
```

The base Sentence Transformer is loaded first.

We then access its underlying Hugging Face transformer and use `PeftModel` to load
the saved adapter.

Finally, the adapted transformer is placed back inside the Sentence Transformer
pipeline.

The adapter is loaded for inference only. No training occurs in this notebook.

In [11]:
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

lora_model = SentenceTransformer(
    MODEL_NAME,
    device=DEVICE
)

lora_model.max_seq_length = MAX_SEQ_LENGTH

transformer_module = lora_model._first_module()
base_hf_model = transformer_module.auto_model

adapted_hf_model = PeftModel.from_pretrained(
    base_hf_model,
    BEST_ADAPTER_DIR,
    is_trainable=False,
)

transformer_module.auto_model = adapted_hf_model

print("Best LoRA adapter loaded.")
print("Adapter path:", BEST_ADAPTER_DIR.resolve())

if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"Free VRAM: {free / 1024**3:.2f} GB")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 15338.60it/s]


Best LoRA adapter loaded.
Adapter path: C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\models\bge-m3-legal-pt-lora\best
Free VRAM: 2.79 GB


## 12. Evaluate the fine-tuned model on the same test set

Nothing about the evaluation protocol changes here.

The same:

- passages;
- queries;
- qrels;
- Top-K;
- normalisation;
- metrics

are used.

Therefore, differences between this output and the baseline can be attributed to
the selected LoRA adaptation under the defined experimental setup.

In [12]:
print("Running LoRA test retrieval...")

lora_results, lora_seconds = evaluate_model(
    lora_model,
    passages,
    test_queries,
    relevant_test,
)

print(f"LoRA evaluation time: {lora_seconds:.1f} s")

lora_summary = aggregate_results(
    lora_results,
    "lora"
)

display(lora_summary)

Running LoRA test retrieval...


Batches: 100%|██████████| 141/141 [00:06<00:00, 22.22it/s]


LoRA evaluation time: 72.4 s


,model,query_group,queries,recall_at_1,recall_at_5,recall_at_10,mrr_at_10,ndcg_at_10
0,lora,all,561,0.607442,0.738191,0.766028,0.737160,0.744312
1,lora,semantic,420,0.806012,0.970734,0.998413,0.971732,0.978920
2,lora,reference,141,0.015957,0.045508,0.073818,0.038436,0.045480


## 13. Final baseline vs. LoRA comparison

We now place the two held-out test results side by side.

For every metric:

```text
absolute delta = LoRA - baseline
```

We also calculate relative change:

```text
relative change (%) =
    (LoRA - baseline) / baseline × 100
```

These two quantities must not be confused.

Example:

```text
0.80 → 0.90

absolute gain = +0.10
              = +10 percentage points

relative gain = 12.5%
```

For reporting retrieval metrics, the absolute difference is often the clearest
primary comparison.

In [13]:
summary = pd.concat(
    [baseline_summary, lora_summary],
    ignore_index=True
)

baseline_wide = (
    baseline_summary
    .set_index("query_group")
)

lora_wide = (
    lora_summary
    .set_index("query_group")
)

comparison_rows = []

for query_group in baseline_wide.index.intersection(lora_wide.index):
    row = {
        "query_group": query_group,
        "queries": int(lora_wide.loc[query_group, "queries"]),
    }

    for metric in METRIC_COLUMNS:
        baseline_value = float(
            baseline_wide.loc[query_group, metric]
        )
        lora_value = float(
            lora_wide.loc[query_group, metric]
        )

        delta = lora_value - baseline_value

        relative_pct = (
            (delta / baseline_value) * 100
            if baseline_value != 0
            else np.nan
        )

        row[f"baseline_{metric}"] = baseline_value
        row[f"lora_{metric}"] = lora_value
        row[f"delta_{metric}"] = delta
        row[f"relative_pct_{metric}"] = relative_pct

    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)

display(comparison_df)

,query_group,queries,baseline_recall_at_1,lora_recall_at_1,delta_recall_at_1,relative_pct_recall_at_1,baseline_recall_at_5,lora_recall_at_5,delta_recall_at_5,relative_pct_recall_at_5,...,delta_recall_at_10,relative_pct_recall_at_10,baseline_mrr_at_10,lora_mrr_at_10,delta_mrr_at_10,relative_pct_mrr_at_10,baseline_ndcg_at_10,lora_ndcg_at_10,delta_ndcg_at_10,relative_pct_ndcg_at_10
0,all,561,0.526084,0.607442,0.081358,15.464762,0.724228,0.738191,0.013963,1.928007,...,-0.023807,-3.014137,0.704021,0.737160,0.033140,4.707180,0.701431,0.744312,0.042881,6.113406
1,semantic,420,0.641660,0.806012,0.164352,25.613540,0.837004,0.970734,0.133730,15.977243,...,0.112731,12.728223,0.824853,0.971732,0.146879,17.806727,0.818472,0.978920,0.160448,19.603382
2,reference,141,0.181816,0.015957,-0.165859,-91.223318,0.388298,0.045508,-0.342790,-88.280061,...,-0.430516,-85.363281,0.344095,0.038436,-0.305660,-88.829909,0.352798,0.045480,-0.307318,-87.108773


## 14. Compact percentage table

This view is intended for presentation and teaching.

The underlying values remain stored as decimals. Formatting them as percentages
does not alter the calculations.

In [14]:
display_columns = ["query_group", "queries"]

formatted = comparison_df[display_columns].copy()

for metric in METRIC_COLUMNS:
    formatted[f"Baseline {metric}"] = (
        comparison_df[f"baseline_{metric}"] * 100
    ).map(lambda x: f"{x:.2f}%")

    formatted[f"LoRA {metric}"] = (
        comparison_df[f"lora_{metric}"] * 100
    ).map(lambda x: f"{x:.2f}%")

    formatted[f"Δ {metric}"] = (
        comparison_df[f"delta_{metric}"] * 100
    ).map(lambda x: f"{x:+.2f} pp")

display(formatted)

,query_group,queries,Baseline recall_at_1,LoRA recall_at_1,Δ recall_at_1,Baseline recall_at_5,LoRA recall_at_5,Δ recall_at_5,Baseline recall_at_10,LoRA recall_at_10,Δ recall_at_10,Baseline mrr_at_10,LoRA mrr_at_10,Δ mrr_at_10,Baseline ndcg_at_10,LoRA ndcg_at_10,Δ ndcg_at_10
0,all,561,52.61%,60.74%,+8.14 pp,72.42%,73.82%,+1.40 pp,78.98%,76.60%,-2.38 pp,70.40%,73.72%,+3.31 pp,70.14%,74.43%,+4.29 pp
1,semantic,420,64.17%,80.60%,+16.44 pp,83.70%,97.07%,+13.37 pp,88.57%,99.84%,+11.27 pp,82.49%,97.17%,+14.69 pp,81.85%,97.89%,+16.04 pp
2,reference,141,18.18%,1.60%,-16.59 pp,38.83%,4.55%,-34.28 pp,50.43%,7.38%,-43.05 pp,34.41%,3.84%,-30.57 pp,35.28%,4.55%,-30.73 pp


## 15. Results by legal document

A global improvement does not guarantee that every legal document benefits equally.

We therefore compare baseline and LoRA performance by:

```text
document × query type
```

This can reveal:

- documents with strong gains;
- documents with little change;
- documents where performance degrades;
- whether the effect is concentrated in semantic queries.

Small document-level query counts should be interpreted cautiously because their
metrics are less stable.

In [15]:
baseline_by_document = aggregate_by_document(
    baseline_results,
    "baseline"
)

lora_by_document = aggregate_by_document(
    lora_results,
    "lora"
)

by_document = baseline_by_document.merge(
    lora_by_document,
    on=["document", "query_type"],
    suffixes=("_baseline", "_lora")
)

for metric in METRIC_COLUMNS:
    by_document[f"delta_{metric}"] = (
        by_document[f"{metric}_lora"]
        - by_document[f"{metric}_baseline"]
    )

semantic_by_document = (
    by_document[
        by_document["query_type"] == "semantic"
    ]
    .sort_values("delta_mrr_at_10", ascending=False)
)

display(semantic_by_document)

,model_baseline,document,query_type,queries_baseline,recall_at_1_baseline,recall_at_5_baseline,recall_at_10_baseline,mrr_at_10_baseline,ndcg_at_10_baseline,model_lora,...,recall_at_1_lora,recall_at_5_lora,recall_at_10_lora,mrr_at_10_lora,ndcg_at_10_lora,delta_recall_at_1,delta_recall_at_5,delta_recall_at_10,delta_mrr_at_10,delta_ndcg_at_10
1,baseline,NaN,semantic,420,0.64166,0.837004,0.885681,0.824853,0.818472,lora,...,0.806012,0.970734,0.998413,0.971732,0.97892,0.164352,0.13373,0.112731,0.146879,0.160448


## 16. Compare individual query behaviour

Aggregate metrics answer:

> Did the model improve on average?

Per-query analysis answers:

> Which queries improved, stayed unchanged or became worse?

We use the difference in reciprocal rank:

```text
ΔRR = LoRA reciprocal rank - baseline reciprocal rank
```

A positive value means the first relevant result moved upward.

A negative value means it moved downward.

This is useful because an average improvement can coexist with a small number of
degraded queries that deserve inspection.

In [16]:
per_query = baseline_results[
    [
        "query_id",
        "query",
        "query_type",
        "document",
        "article_id",
        "relevant_ids",
        "ranked_ids",
        "ranked_scores",
        *METRIC_COLUMNS,
    ]
].merge(
    lora_results[
        [
            "query_id",
            "ranked_ids",
            "ranked_scores",
            *METRIC_COLUMNS,
        ]
    ],
    on="query_id",
    suffixes=("_baseline", "_lora")
)

for metric in METRIC_COLUMNS:
    per_query[f"delta_{metric}"] = (
        per_query[f"{metric}_lora"]
        - per_query[f"{metric}_baseline"]
    )

semantic_per_query = per_query[
    per_query["query_type"] == "semantic"
].copy()

semantic_per_query["outcome"] = np.select(
    [
        semantic_per_query["delta_mrr_at_10"] > 0,
        semantic_per_query["delta_mrr_at_10"] < 0,
    ],
    [
        "improved",
        "degraded",
    ],
    default="unchanged",
)

print(
    semantic_per_query["outcome"].value_counts()
)

display(
    semantic_per_query[
        [
            "query_id",
            "document",
            "mrr_at_10_baseline",
            "mrr_at_10_lora",
            "delta_mrr_at_10",
            "outcome",
        ]
    ]
    .sort_values("delta_mrr_at_10", ascending=False)
    .head(20)
)

outcome
unchanged    318
improved      93
degraded       9
Name: count, dtype: int64


,query_id,document,mrr_at_10_baseline,mrr_at_10_lora,delta_mrr_at_10,outcome
27,d6f573c1db173969,None,0.0,1.0,1.0,improved
349,9f6793ce031ff0c6,None,0.0,1.0,1.0,improved
118,a9b0b2614ea6b361,None,0.0,1.0,1.0,improved
153,2a89572da3c9af64,None,0.0,1.0,1.0,improved
25,061da28f4713cfbb,None,0.0,1.0,1.0,improved
154,07731e1e8e4713d3,None,0.0,1.0,1.0,improved
175,6a3c01470761d7c2,None,0.0,1.0,1.0,improved
174,e47859ea2ccdc45b,None,0.0,1.0,1.0,improved
173,da36b78147f0eea8,None,0.0,1.0,1.0,improved
186,1fb340e2e93c990a,None,0.0,1.0,1.0,improved


## 17. Qualitative ranking inspection

Metrics should be complemented by examples.

The helper below displays:

- the query;
- relevant passage IDs;
- baseline Top-K;
- LoRA Top-K;
- similarity scores;
- relevance markers;
- passage text.

This allows us to see *how* LoRA changed the ranking.

A useful improved example is one where a relevant legal provision moves from a low
rank to rank 1.

A useful degraded example is one where the baseline ranked the relevant passage
higher than the adapted model.

Qualitative analysis can also expose ambiguous queries or incomplete qrels.

In [17]:
def inspect_comparison(row, top_n=5):
    print("=" * 100)
    print("QUERY")
    print(row["query"])
    print()
    print("Document:", row["document"])
    print("Relevant IDs:", row["relevant_ids"])
    print(
        "Baseline MRR:",
        f'{row["mrr_at_10_baseline"]:.4f}',
        "| LoRA MRR:",
        f'{row["mrr_at_10_lora"]:.4f}',
    )

    relevant = set(row["relevant_ids"])

    for label in ["baseline", "lora"]:
        print()
        print(label.upper())
        print("-" * 100)

        ranked_ids = row[f"ranked_ids_{label}"]
        ranked_scores = row[f"ranked_scores_{label}"]

        for rank, (passage_id, score) in enumerate(
            zip(ranked_ids[:top_n], ranked_scores[:top_n]),
            start=1,
        ):
            passage = passage_by_id[passage_id]
            marker = "✓ RELEVANT" if passage_id in relevant else ""

            print(
                f"Rank {rank:>2} | score={score:.4f} | "
                f"{passage_id} {marker}"
            )
            print(passage["text"][:700])
            print()

## 18. Inspect the strongest semantic improvements

These are the semantic queries with the largest positive change in reciprocal rank.

They are useful examples for explaining what the fine-tuning actually changed in
the embedding space.

In [18]:
improved_examples = (
    semantic_per_query[
        semantic_per_query["outcome"] == "improved"
    ]
    .sort_values("delta_mrr_at_10", ascending=False)
)

if improved_examples.empty:
    print("No semantic query improved in MRR@10.")
else:
    for _, row in improved_examples.head(3).iterrows():
        inspect_comparison(row, top_n=5)

QUERY
O que prevê o CIMI relativamente a Publicação?

Document: None
Relevant IDs: ['cimi_block_02_art_88_o']
Baseline MRR: 0.0000 | LoRA MRR: 1.0000

BASELINE
----------------------------------------------------------------------------------------------------
Rank  1 | score=0.5469 | cimi_block_02_art_124_o 
Documento: CIMI
Estrutura: Chapter XIII — Fiscalização
Artigo 124.º — Entidades públicas
1 - As entidades públicas, ou que desempenhem funções públicas, que intervenham em actos
relativos à constituição, transmissão, registo ou litígio de direitos sobre prédios, devem exigir a
exibição de documento comprovativo da inscrição do prédio na matriz ou, sendo omisso, de que
foi apresentada a declaração para inscrição.
2 - Sempre que o cumprimento do disposto no número anterior se mostre impossível, faz-se
expressa menção do facto e das razões dessa impossibilidade, devendo comunicar-se tal facto
ao serviço de finanças da área da situação dos prédios.

Rank  2 | score=0.5202 | cimi_block

## 19. Inspect semantic degradations

A credible evaluation should not show only successful examples.

If some queries become worse after fine-tuning, we inspect them explicitly.

Possible explanations include:

- the adapter over-specialised toward patterns in the training triplets;
- the hard-negative supervision created an undesirable boundary;
- the query is ambiguous;
- several legal provisions are semantically valid but the qrels contain only some;
- the adapted embedding space improved globally while sacrificing a particular
  local distinction.

Degraded cases are especially useful for deciding what should change in a future
dataset or training run.

In [19]:
degraded_examples = (
    semantic_per_query[
        semantic_per_query["outcome"] == "degraded"
    ]
    .sort_values("delta_mrr_at_10")
)

if degraded_examples.empty:
    print("No semantic query degraded in MRR@10.")
else:
    for _, row in degraded_examples.head(3).iterrows():
        inspect_comparison(row, top_n=5)

QUERY
O que prevê o CIRC relativamente a (Artigo aditado pela Lei n.o 24/2020, de 6 de julho)?

Document: None
Relevant IDs: ['circ_block_01_art_68_o_d__chunk_01', 'circ_block_01_art_68_o_d__chunk_02']
Baseline MRR: 1.0000 | LoRA MRR: 0.1250

BASELINE
----------------------------------------------------------------------------------------------------
Rank  1 | score=0.5845 | circ_block_01_art_68_o_d__chunk_02 ✓ RELEVANT
Documento: CIRC
Estrutura: Chapter III — Determinação da matéria coletável | Section VI — Disposições comuns e diversas | Subsection I — Correções para efeitos da determinação da matéria coletável
Artigo 68.º-D — (Artigo aditado pela Lei n.o 24/2020, de 6 de julho)
2 - Sem prejuízo do disposto no número seguinte, os artigos 68.o-A, 68.o-B e 68.o-D do Código do IRC, com a
redação introduzida pela presente lei, são aplicáveis aos períodos de tributação com início em, ou após, 1 de
janeiro de 2020 e o artigo 68.o-C do Código do IRC, com a redação introduzida pela presente 

## 20. Save detailed final results

We preserve both aggregate and per-query results.

JSONL is used for detailed rankings because list-valued fields such as ranked
passage IDs and similarity scores are naturally represented in JSON.

CSV is used for tabular summaries.

These files make the final conclusions auditable without rerunning both embedding
models.

In [20]:
def dataframe_to_jsonl(df, path):
    with path.open("w", encoding="utf-8") as f:
        for record in df.to_dict(orient="records"):
            f.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                    default=str,
                )
                + "\n"
            )


BASELINE_DETAILS_PATH = RESULTS_DIR / "baseline_test_results.jsonl"
LORA_DETAILS_PATH = RESULTS_DIR / "lora_test_results.jsonl"
PER_QUERY_PATH = RESULTS_DIR / "test_query_comparison.jsonl"

SUMMARY_PATH = RESULTS_DIR / "test_summary.csv"
COMPARISON_PATH = RESULTS_DIR / "test_comparison.csv"
BY_DOCUMENT_PATH = RESULTS_DIR / "test_by_document.csv"

dataframe_to_jsonl(
    baseline_results,
    BASELINE_DETAILS_PATH
)

dataframe_to_jsonl(
    lora_results,
    LORA_DETAILS_PATH
)

dataframe_to_jsonl(
    per_query,
    PER_QUERY_PATH
)

summary.to_csv(
    SUMMARY_PATH,
    index=False
)

comparison_df.to_csv(
    COMPARISON_PATH,
    index=False
)

by_document.to_csv(
    BY_DOCUMENT_PATH,
    index=False
)

print(BASELINE_DETAILS_PATH.resolve())
print(LORA_DETAILS_PATH.resolve())
print(PER_QUERY_PATH.resolve())
print(SUMMARY_PATH.resolve())
print(COMPARISON_PATH.resolve())
print(BY_DOCUMENT_PATH.resolve())

C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\results\final_evaluation\baseline_test_results.jsonl
C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\results\final_evaluation\lora_test_results.jsonl
C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\results\final_evaluation\test_query_comparison.jsonl
C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\results\final_evaluation\test_summary.csv
C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\results\final_evaluation\test_comparison.csv
C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\results\final_evaluation\test_by_document.csv


## 21. Save the final evaluation report

The final JSON report records:

- model and adapter;
- selected epoch;
- dataset sizes;
- evaluation configuration;
- execution times;
- baseline metrics;
- LoRA metrics;
- metric differences;
- semantic query outcome counts.

This becomes the compact machine-readable record of the final experiment.

In [21]:
def records_for_json(df):
    return json.loads(
        df.to_json(
            orient="records",
            force_ascii=False
        )
    )


outcome_counts = (
    semantic_per_query["outcome"]
    .value_counts()
    .to_dict()
)

report = {
    "experiment": "final_test_evaluation",
    "base_model": MODEL_NAME,
    "adapter_path": str(BEST_ADAPTER_DIR),
    "selected_epoch": finetuning_metadata["best_epoch"],
    "device": DEVICE,
    "max_seq_length": MAX_SEQ_LENGTH,
    "encode_batch_size": ENCODE_BATCH_SIZE,
    "top_k": TOP_K,
    "passages": len(passages),
    "test_queries": len(test_queries),
    "test_qrels": len(test_qrels),
    "test_query_types": dict(
        Counter(q["query_type"] for q in test_queries)
    ),
    "baseline_seconds": baseline_seconds,
    "lora_seconds": lora_seconds,
    "summary": records_for_json(summary),
    "comparison": records_for_json(comparison_df),
    "semantic_query_outcomes": {
        str(k): int(v)
        for k, v in outcome_counts.items()
    },
}

REPORT_PATH = RESULTS_DIR / "final_evaluation_report.json"

with REPORT_PATH.open("w", encoding="utf-8") as f:
    json.dump(
        report,
        f,
        ensure_ascii=False,
        indent=2
    )

print(REPORT_PATH.resolve())

C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\results\final_evaluation\final_evaluation_report.json


## 22. Interpretation of the observed test results

The held-out test set provides a more nuanced result than the validation stage.

The LoRA adaptation is **highly effective for semantic retrieval**, which was the
task used during fine-tuning, but it substantially degrades **reference-query
retrieval**. This distinction is essential when interpreting the experiment.

### 1. Overall results

Across all 561 test queries:

| Metric | Baseline | LoRA | Absolute change |
|---|---:|---:|---:|
| Recall@1 | 52.61% | **60.74%** | **+8.14 pp** |
| Recall@5 | 72.42% | **73.82%** | **+1.40 pp** |
| Recall@10 | **78.98%** | 76.60% | **−2.38 pp** |
| MRR@10 | 70.40% | **73.72%** | **+3.31 pp** |
| nDCG@10 | 70.14% | **74.43%** | **+4.29 pp** |

If only the global averages were inspected, the conclusion would appear moderately
positive: LoRA improves Recall@1, Recall@5, MRR@10 and nDCG@10.

However, this global average combines two query types that behave very differently.

---

### 2. Semantic queries — the target task

For the 420 semantic test queries, the improvement is substantial:

| Metric | Baseline | LoRA | Absolute change |
|---|---:|---:|---:|
| Recall@1 | 64.17% | **80.60%** | **+16.44 pp** |
| Recall@5 | 83.70% | **97.07%** | **+13.37 pp** |
| Recall@10 | 88.57% | **99.84%** | **+11.27 pp** |
| MRR@10 | 82.49% | **97.17%** | **+14.69 pp** |
| nDCG@10 | 81.85% | **97.89%** | **+16.04 pp** |

This is the strongest result of the experiment.

Semantic `MRR@10` increases from:

```text
0.8249 → 0.9717
```

an absolute improvement of approximately **14.69 percentage points**.

Semantic `Recall@1` also rises from **64.17% to 80.60%**, meaning that relevant
content is much more frequently placed immediately at the top of the ranking.

`Recall@10 = 99.84%` indicates that almost all relevant semantic content is
retrieved within the first ten results.

Most importantly, these gains occur on the **held-out test set**. The test articles
were not used for training or checkpoint selection.

The experiment therefore provides evidence that the LoRA adaptation learned
retrieval patterns that generalise beyond the training and validation articles.

---

### 3. Per-query semantic behaviour

The 420 semantic test queries are distributed as:

```text
Improved    93
Unchanged  318
Degraded     9
```

Therefore:

- approximately **22.1%** of semantic queries improve in MRR@10;
- approximately **75.7%** remain unchanged;
- approximately **2.1%** degrade.

The improvement is therefore not produced by a broad trade-off in which many
queries become worse. Only a small minority of semantic queries degrade.

Several examples show a particularly strong change:

```text
Baseline MRR@10 = 0
LoRA MRR@10     = 1
```

In these cases, the baseline failed to place any relevant passage within the first
ten results, while the adapted model moved a relevant passage directly to rank 1.

The CIMI example displayed above illustrates this clearly:

```text
Query:
"O que prevê o CIMI relativamente a Publicação?"

Baseline:
relevant Article 88 not present in Top-10

LoRA:
Article 88 → rank 1
```

This is a concrete example of the embedding space becoming more useful for legal
semantic retrieval.

---

### 4. Reference queries — substantial degradation

The behaviour is very different for the 141 reference queries:

| Metric | Baseline | LoRA | Absolute change |
|---|---:|---:|---:|
| Recall@1 | **18.18%** | 1.60% | **−16.59 pp** |
| Recall@5 | **38.83%** | 4.55% | **−34.28 pp** |
| Recall@10 | **50.43%** | 7.38% | **−43.05 pp** |
| MRR@10 | **34.41%** | 3.84% | **−30.57 pp** |
| nDCG@10 | **35.28%** | 4.55% | **−30.73 pp** |

This degradation is not a minor fluctuation.

Reference `MRR@10` falls from:

```text
0.3441 → 0.0384
```

and Recall@10 falls from **50.43% to only 7.38%**.

This behaviour is consistent with the training design: notebook 06 deliberately
trained LoRA **only on semantic queries**.

The adapter has therefore specialised the embedding space toward semantic matching,
but this specialisation has damaged the lexical/reference behaviour needed for
queries that depend strongly on article identifiers, headings or explicit legal
references.

This is an important experimental result rather than something that should be
hidden by the global average.

---

### 5. Example of reference-query degradation

One displayed example concerns:

```text
"O que prevê o CIRC relativamente a
(Artigo aditado pela Lei n.º 24/2020, de 6 de julho)?"
```

The baseline retrieves a relevant Article 68.º-D chunk at **rank 1**:

```text
Baseline MRR@10 = 1.0000
```

After LoRA, several highly similar provisions from Articles 68.º-A and 68.º-B move
above the relevant Article 68.º-D:

```text
LoRA MRR@10 = 0.1250
```

This example helps explain the aggregate reference-query degradation.

The adapted model has become very sensitive to the shared semantic context of the
68.º-A–68.º-D provisions, but less effective at preserving the precise reference
signal needed to distinguish the requested provision.

---

### 6. What can we conclude?

The correct conclusion depends on the intended use of the model.

If the objective is specifically:

> **semantic retrieval over Portuguese legal documents**

then the fine-tuning experiment is successful.

The held-out semantic test results show large improvements across every metric,
including:

```text
MRR@10    +14.69 pp
nDCG@10   +16.04 pp
Recall@1  +16.44 pp
Recall@10 +11.27 pp
```

If the objective is instead to create **one embedding model that performs both
semantic and explicit-reference retrieval**, the current adapter is not yet an
adequate final model.

Its semantic gains come with a severe loss on reference queries.

---

### 7. Methodological implication for the next experiment

The result suggests that the next training version should investigate a
**mixed-objective or mixed-query training dataset**.

Instead of:

```text
Training
└── semantic queries only
```

a future experiment could use:

```text
Training
├── semantic queries
└── reference queries
```

with controlled sampling so that semantic adaptation is retained without destroying
reference retrieval.

Another practical architecture is to treat the two retrieval problems separately:

```text
User query
   ↓
Query analysis
   ├── semantic query  → dense embedding retrieval
   └── explicit legal reference
                        → lexical / metadata / hybrid retrieval
```

For legal RAG, this second design may be particularly appropriate because a query
containing an explicit article number should not necessarily rely exclusively on
semantic similarity.

---

## Final conclusion

The experiment demonstrates that **LoRA fine-tuning of BGE-M3 with hard-negative
semantic triplets can substantially improve semantic retrieval in Portuguese legal
documents**.

The improvement generalises to held-out test articles:

```text
Semantic MRR@10
Baseline BGE-M3 : 0.8249
BGE-M3 + LoRA   : 0.9717
Difference      : +0.1469
```

At the same time, the experiment reveals an important limitation: semantic-only
fine-tuning strongly degrades explicit reference retrieval.

The final result should therefore be described as **successful domain adaptation
for semantic legal retrieval**, rather than as an unconditional improvement of
BGE-M3 across every type of legal query.

## 23. Output structure and next experiment

The final evaluation files are stored under:

```text
results/
└── final_evaluation/
    ├── baseline_test_results.jsonl
    ├── lora_test_results.jsonl
    ├── test_query_comparison.jsonl
    ├── test_summary.csv
    ├── test_comparison.csv
    ├── test_by_document.csv
    └── final_evaluation_report.json
```

The first complete experiment is now closed:

```text
Legal PDFs
   ↓
Structured articles
   ↓
Retrieval dataset
   ↓
Baseline BGE-M3
   ↓
Hard-negative mining
   ↓
Semantic-only LoRA fine-tuning
   ↓
Validation checkpoint selection
   ↓
Held-out test evaluation
   ↓
Strong semantic gains
   +
Reference-query degradation
```

This result gives the next experiment a clear research question:

> **Can we preserve the semantic gains of LoRA while recovering reference-query
> performance?**

Possible next steps include mixed semantic/reference fine-tuning, a hybrid
dense–lexical retrieval architecture, or explicit routing of article-reference
queries to metadata-aware retrieval.

The current experiment should be preserved unchanged as the baseline for those
future variants.